In [1]:
from ctypes import CDLL, c_uint32
from pathlib import Path

library_path = Path.cwd() / "slascone-client" / "SlasconeClient.so"
if not library_path.is_file():
    raise FileNotFoundError(f"Slascone client library not found: {library_path}")

slascone_client = CDLL(str(library_path))
slascone_client.slascone_client_abi_version.restype = c_uint32
slascone_client.slascone_client_abi_version.argtypes = []
abi_version = slascone_client.slascone_client_abi_version()
print(f"Loaded {library_path} (ABI version {abi_version})")

Loaded /workspace/slascone-client/SlasconeClient.so (ABI version 2)


### Using the Python client

`SlasconeClient` owns the native handle and closes it when the `with` block ends. The setters and the license check return a `SlasconeStatus`.

The ABI probe above only checks that the library loads. The bundled `.so` was observed to crash in `client_create` in the Jupyter Linux image, even from a C program. Use a working ABI 1 build via `library_path` before running this example.

```python
from pathlib import Path
from slascone_client import SlasconeClient, SlasconeStatus

with SlasconeClient("your-product-id", library_path="/path/to/SlasconeClient.so") as client:
    for configure, value in (
        (client.set_client_id, "your-client-id"),
        (client.set_software_release, "1.0"),
        (client.set_pem_key, Path("public-key.pem").read_text(encoding="utf-8")),
    ):
        status = configure(value)
        if status != SlasconeStatus.OK:
            raise RuntimeError(f"{configure.__name__}: {status.name}")

    status = client.read_offline_license("license.xml")
    if status != SlasconeStatus.OK:
        raise RuntimeError(f"read_offline_license: {status.name}")

    print(client.check_offline_license().name)  # Inline activation
```

To use a separate activation file, pass its path to `check_offline_license` after loading the license with `read_offline_license`.

#### Extract the container ID from `/proc/self/mountinfo`

In [3]:
with open('/proc/self/mountinfo') as file:
    line = file.readline().strip()
    while line:
        if '/docker/containers/' in line:
            containerID = line.split('/docker/containers/')[-1]
            containerID = containerID.split('/')[0]
            print(f"Container ID: {containerID}")
        line = file.readline().strip()

Container ID: 090a258e5d0949f1cf12247ccea36a8bf82bc2c538d8b6c5757498a653a31284
Container ID: 090a258e5d0949f1cf12247ccea36a8bf82bc2c538d8b6c5757498a653a31284
Container ID: 090a258e5d0949f1cf12247ccea36a8bf82bc2c538d8b6c5757498a653a31284


#### Get a offline activation file

See [PERMANENTLY OFFLINE SCENARIOS - LICENSE FILES](https://support.slascone.com/hc/en-us/articles/4412248454161-PERMANENTLY-OFFLINE-SCENARIOS-LICENSE-FILES)

API endpoint

```
GET /isv/{isv_id}/provisioning/activations/offline
```

Parameters

| Parameter | Type | Mandatory |
|-----------|------|-----------|
| product_id | string | yes |
| license_key | string | yes |
| client_id | string | yes |
| client_name | string | no |
| client_description | string | no |
| software_version | string | yes |
| tags | array<string> | no |
| file_name | string | no |



In [10]:
import json
import os
from pathlib import Path

import requests

base_url = "https://api.slascone.com"
isv_id = "2af5fe02-6207-4214-946e-b00ac5309f53"
product_id = "47df1df5-bbc8-4b1b-a185-58ddfb1d3271"
license_key = "1eb98e63-1146-4497-a4a7-7e4617506d27"
software_version = "26.3"
client_id = containerID
client_name = f"SLASCONE Python SDK ({client_id[:12]})"

url = f"{base_url}/api/v2/isv/{isv_id}/provisioning/activations/offline"
params = {
    "product_id": product_id,
    "license_key": license_key,
    "client_id": client_id,
    "client_name": client_name,
#    "software_version": software_version,
    "file_name": "ActivationFile.xml"
}
headers = {"Accept": "application/xml"}

response = requests.get(url, params=params, headers=headers, timeout=30)

if response.status_code == 200:
    offline_license_path = Path.cwd() / "assets" / f"ActivationFile.xml"
    offline_license_path.write_bytes(response.content)
    print(f"Offline license saved to {offline_license_path}")
elif response.status_code == 409:
    print("Conflict detected:")
    print(response.content.decode('utf-8'))
else:
    print(f"Unexpected response ({response.status_code}): {response.content.decode('utf-8')}")    




Offline license saved to /workspace/assets/ActivationFile.xml


#### Load and initialze the library

In [15]:
from pathlib import Path
from slascone_client import SlasconeClient, SlasconeStatus

library_path = Path.cwd() / "slascone-client" / "SlasconeClient.so"

public_key = """-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAwpigzm+cZIyw6x253YRD
mroGQyo0rO9qpOdbNAkE/FMSX+At5CQT/Cyr0eZTo2h+MO5gn5a6dwg2SYB/K1Yt
yuiKqnaEUfoPnG51KLrj8hi9LoZyIenfsQnxPz+r8XGCUPeS9MhBEVvT4ba0x9Ew
R+krU87VqfI3KNpFQVdLPaZxN4STTEZaet7nReeNtnnZFYaUt5XeNPB0b0rGfrps
y7drmZz81dlWoRcLrBRpkf6XrOTX4yFxe/3HJ8mpukuvdweUBFoQ0xOHmG9pNQ31
AHGtgLYGjbKcW4xYmpDGl0txfcipAr1zMj7X3oCO9lHcFRnXdzx+TTeJYxQX2XVb
hQIDAQAB
-----END PUBLIC KEY-----"""

license_file = Path.cwd() / "assets" / "License_1eb98e63-1146-4497-a4a7-7e4617506d27.xml"
activation_file = Path.cwd() / "assets" / "ActivationFile.xml"

client = SlasconeClient(product_id, library_path=library_path)

for configure, value in (
    (client.set_client_id, containerID),
    (client.set_software_release, "26.3"),
    (client.set_pem_key, public_key),
):
    status = configure(value)
    if status != SlasconeStatus.OK:
        raise RuntimeError(f"{configure.__name__}: {status.name}")

status = client.read_offline_license(license_file)
if status != SlasconeStatus.OK:
    raise RuntimeError(f"read_offline_license: {status.name}")

        

#### Check license validity and activation

In [16]:
print("License check: ", client.check_offline_license(activation_file).name)

License check:  OK


#### Check features availability

In [17]:
for feature_id, feature_name in (
    ("03fc55e6-51de-4bd4-bef3-55ef65393245", "Scheduled Report Distribution"),
    ("48be7014-afe8-420a-99a8-66d4fa2c9e86", "Drill-Through"),
    ("7b5f123c-a6cb-4292-8554-9581f9728e8c", "Print"),
    ("827b9df9-e326-4980-b127-9fd8d3b66f68", "Dashboards"),
    ("8aa72096-5b3e-4119-01f3-08dd65e33461", "New feature in 2025"),
    ("8f2fa738-3e5c-4b5e-af1a-b238136ca54d", "Powerpoint Export"),
    ("9ccb53bd-ae6b-45ef-a482-7a8f6f2e5845", "SAP Interface"),
    ("9d645a6d-72b3-4b47-43dd-08de83ec2f07", "New Feature 2027"),
    ("c0f282a5-f6c2-45fe-a5a6-ad715332825f", "Orders"),
    ("cb112265-b800-485c-b152-afdcababc1ad", "PDF-Forms"),
    ("d277c8d7-fac5-4c0e-882c-ad64a510d869", "Commenting"),
    ("f8c31e99-778a-4537-a6f9-6fbe87c085b1", "Project")
):
    status, is_activated = client.check_license_feature(feature_id)
    if status != SlasconeStatus.OK:
        raise RuntimeError(f"check_license_feature: {status}")
    else:
        if is_activated:
            print(f"License feature check passed; feature '{feature_name}' is activated")
        else:
            print(f"License feature check passed; feature '{feature_name}' is not activated")


License feature check passed; feature 'Scheduled Report Distribution' is activated
License feature check passed; feature 'Drill-Through' is not activated
License feature check passed; feature 'Print' is not activated
License feature check passed; feature 'Dashboards' is activated
License feature check passed; feature 'New feature in 2025' is activated
License feature check passed; feature 'Powerpoint Export' is activated
License feature check passed; feature 'SAP Interface' is activated
License feature check passed; feature 'New Feature 2027' is not activated
License feature check passed; feature 'Orders' is activated
License feature check passed; feature 'PDF-Forms' is activated
License feature check passed; feature 'Commenting' is activated
License feature check passed; feature 'Project' is activated
